In [1]:
import pandas as pd
# from sqlalchemy import create_engine
# engine = create_engine("mysql+pymysql://tikkle_admin:tikkle_pass1234@db:3306/tikkle_db")

df=pd.read_csv('./pantry_end.csv')
df.reset_index(drop=True)
df['ingredient_id'] = df.index + 1
cols  = ['ingredient_id'] + [col for col in df.columns if col != 'ingredient_id']
df= df[cols]
df.to_csv('pantry_end.csv',index=False)

In [2]:
df

,ingredient_id,ingredient_name,category,storage_type,expiry_date
0,1,LA갈비,육류,냉장,0
1,2,가는파,기타,실온,0
2,3,가다랑어포,기타,실온,0
3,4,가다랑이포,기타,실온,0
4,5,가래떡,기타,실온,0
...,...,...,...,...,...
1484,1485,흰색,기타,실온,0
1485,1486,흰설탕,기타,실온,0
1486,1487,흰콩,기타,실온,0
1487,1488,흰후추,기타,실온,0


In [3]:
import pandas as pd
import re

df = pd.read_csv("pantry_end.csv")

def normalize(text):
    text = str(text).strip()
    text = re.sub(r"\([^)]*\)", "", text)  # 괄호 제거
    text = re.sub(r"[,\./·]", " ", text)
    text = re.sub(r"(조금|약간|적당량|기호껏|취향껏|생략가능|선택)", "", text)
    text = re.sub(r"\s+", "", text)
    return text

MEAT = [
    "소고기","쇠고기","우고기","소","우","돼지고기","돼기고기","돼지","돈육",
    "닭고기","닭","오리","양고기","양","갈비","삼겹","목살","등심","안심",
    "사골","햄","베이컨","소시지","훈제오리","닭날개","닭다리","닭가슴살"
]

SEAFOOD = [
    "가다랑어","가다랑이","가쓰오","참치","연어","고등어","갈치","조기","명태","대구",
    "황태","북어","코다리","아귀","아구","메기","잉어","장어","새우","건새우",
    "오징어","문어","낙지","쭈꾸미","꼴뚜기","조개","바지락","꼬막","굴","전복",
    "관자","가리비","골뱅이","게","꽃게","대게","멸치","디포리","다슬기","미역",
    "다시마","김","파래","매생이","어묵","맛살","젓갈","새우젓","액젓"
]

DAIRY = [
    "우유","치즈","버터","요거트","요구르트","생크림","휘핑크림",
    "크림치즈","연유","분유","마스카르포네","리코타","모짜렐라",
    "체다","파마산","스트링치즈"
]

FRESH = [
    "파","대파","쪽파","실파","양파","마늘","생강","감자","고구마","무","당근","오이",
    "호박","애호박","단호박","배추","양배추","적양배추","상추","꽃상추","로메인",
    "깻잎","시금치","아욱","미나리","달래","냉이","고사리","더덕","연근","우엉",
    "근대","갓","고수","고수잎","세발나물","호박잎","콩나물","숙주","버섯","표고",
    "느타리","팽이","새송이","토마토","방울토마토","체리토마토","가지","브로콜리",
    "피망","파프리카","고추","청양고추","풋고추","영콘","그린빈","껍질콩",
    "사과","배","포도","오렌지","레몬","라임","귤","감","복숭아","자두","블루베리",
    "딸기","바나나","키위","아보카도","대추"
]

FROZEN_HINT = [
    "냉동","얼린","동태","만두","너겟","돈까스","튀김"
]

def classify_category(name):
    n = normalize(name)

    if any(k in n for k in DAIRY):
        return "유제품"
    if any(k in n for k in MEAT):
        return "육류"
    if any(k in n for k in SEAFOOD):
        return "해산물"
    if any(k in n for k in FRESH):
        return "신선식품"
    return "기타"

def classify_storage(name, category):
    n = normalize(name)

    if any(k in n for k in FROZEN_HINT):
        return "냉동"

    # 냉동 명시가 없으면 기본 냉장
    # (단, 이 데이터는 실온 재료가 많아서 냉장 쏠림은 어느 정도 남음)
    return "냉장"

df["category"] = df["ingredient_name"].apply(classify_category)
df["storage_type"] = df.apply(lambda x: classify_storage(x["ingredient_name"], x["category"]), axis=1)

print(df["category"].value_counts())
print(df["storage_type"].value_counts())

category
기타      752
신선식품    317
육류      221
해산물     151
유제품      48
Name: count, dtype: int64
storage_type
냉장    1472
냉동      17
Name: count, dtype: int64


In [4]:
import pandas as pd
import re

df = pd.read_csv("pantry_end.csv")

def normalize(text):
    text = str(text).strip()
    text = re.sub(r"\([^)]*\)", "", text)
    text = re.sub(r"[\[\],./·]", " ", text)
    text = re.sub(r"(약간|적당량|조금|취향껏|기호껏|선택|약간의)", "", text)
    text = re.sub(r"\s+", "", text)
    return text

MEAT = [
    "소고기","쇠고기","우둔","우삼겹","차돌박이","대패","갈비","사태","양지","등심",
    "돼지고기","돈육","돼지","삼겹","목살","앞다리","뒷다리","항정살","가브리살",
    "닭고기","닭","닭가슴살","닭다리","닭날개","훈제오리","오리","양고기",
    "햄","베이컨","소시지","스팸","미트볼","떡갈비"
]

SEAFOOD = [
    "새우","건새우","오징어","문어","낙지","쭈꾸미","꼴뚜기",
    "조개","바지락","꼬막","굴","전복","관자","가리비","홍합","골뱅이","다슬기",
    "게","꽃게","대게",
    "멸치","디포리","북어","황태","코다리","명태","대구","동태","갈치","고등어",
    "연어","참치","꽁치","조기","임연수","삼치","광어","우럭","장어","아귀","가자미",
    "미역","다시마","김","파래","매생이",
    "액젓","새우젓","멸치액젓","까나리액젓","어묵","맛살","게맛살"
]

DAIRY = [
    "우유","치즈","모짜렐라","체다","파마산","크림치즈","리코타",
    "버터","생크림","휘핑크림","요거트","요구르트","연유","분유","요플레"
]

FRESH = [
    "배추","알배추","양배추","적양배추","상추","로메인","깻잎","시금치","청경채","근대","쑥갓",
    "부추","미나리","달래","냉이","아욱","고사리","숙주","콩나물",
    "대파","쪽파","실파","양파","적양파","마늘","생강",
    "감자","고구마","무","당근","연근","우엉",
    "오이","애호박","단호박","호박","가지","브로콜리",
    "버섯","표고","느타리","팽이","새송이","양송이",
    "고추","청양고추","풋고추","홍고추","파프리카","피망",
    "토마토","방울토마토","샐러리","셀러리","비트","아스파라거스","케일",
    "사과","배","귤","오렌지","레몬","라임","포도","딸기","블루베리","바나나","키위","아보카도","복숭아","감",
    "두부","계란","달걀"
]

ROOM_TEMP_HINT = [
    "소스","간장","된장","고추장","쌈장","식초","케첩","마요네즈",
    "밀가루","부침가루","튀김가루","빵가루","전분","분말","가루",
    "설탕","소금","후추","고춧가루","깨","참깨","통깨",
    "참기름","들기름","올리브유","식용유","오일",
    "국수","라면","파스타","스파게티","당면","소면","중면","우동면","건면",
    "참치캔","통조림","캔","시럽","물엿","올리고당","꿀",
    "콩","팥","녹두","찹쌀","멥쌀","쌀","곡물"
]

FROZEN_HINT = [
    "냉동","얼린","냉장아님",  # 혹시 이상값 있으면 제거 가능
    "만두","너겟","돈까스","새우튀김","감자튀김","튀김","핫도그",
    "동태","냉동새우","냉동오징어","냉동연어","아이스크림"
]

def classify_category(name):
    n = normalize(name)

    if "소스" in n:
        return "기타"
    if any(k in n for k in DAIRY):
        return "유제품"
    if any(k in n for k in MEAT):
        return "육류"
    if any(k in n for k in SEAFOOD):
        return "해산물"
    if any(k in n for k in FRESH):
        return "신선식품"
    return "기타"

def classify_storage(name, category):
    n = normalize(name)

    # 1. 냉동 우선
    if any(k in n for k in FROZEN_HINT):
        return "냉동"

    # 2. 실온 우선
    if any(k in n for k in ROOM_TEMP_HINT):
        return "실온"

    # 3. 카테고리 기반 기본값
    if category in ["육류", "해산물", "유제품", "신선식품"]:
        return "냉장"

    # 4. 나머지 기타 기본값
    return "실온"

df["category"] = df["ingredient_name"].apply(classify_category)
df["storage_type"] = df.apply(lambda x: classify_storage(x["ingredient_name"], x["category"]), axis=1)

print(df["category"].value_counts())
print(df["storage_type"].value_counts())

category
기타      861
신선식품    331
해산물     176
육류       72
유제품      49
Name: count, dtype: int64
storage_type
실온    898
냉장    573
냉동     18
Name: count, dtype: int64


In [5]:
df['expiry_date']=0
df

,ingredient_id,ingredient_name,category,storage_type,expiry_date
0,1,LA갈비,육류,냉장,0
1,2,가는파,기타,실온,0
2,3,가다랑어포,기타,실온,0
3,4,가다랑이포,기타,실온,0
4,5,가래떡,기타,실온,0
...,...,...,...,...,...
1484,1485,흰색,기타,실온,0
1485,1486,흰설탕,기타,실온,0
1486,1487,흰콩,기타,실온,0
1487,1488,흰후추,기타,실온,0


In [ ]:
# df.to_csv('pantry_end.csv',index=False)

In [36]:
import pandas as pd 
df_=pd.read_csv('./티끌최종레시피.csv')
# df_=df_[df_['main_ingredients'].notna()].reset_index(drop=True)

# df_.drop(columns='Unnamed: 0',inplace=True)
# df_.to_csv('티끌최종레시피.csv',index=False)
(df_["main_ingredients"].astype(str).str.strip() == "").sum()
df_['main_ingredients'].isnull().sum()

np.int64(0)

In [30]:
df_['main_ingredients'].isnull().sum()

np.int64(0)

In [7]:
import pandas as pd
df=pd.read_csv('./티끌최종레시피.csv')
df[df['recipe_name'].str.contains('황금레시피')]

,recipe_id,recipe_name,cooking_time,difficulty,sub_ingredients,Seasonings,main_ingredients
